[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# The axis Argument


## What you will be able to do

Choose the `axis` of a reduction from the question being asked, predict the shape of the result
before running it, reduce over several dimensions at once, and keep a reduced dimension so the result
can be subtracted from the array it came from.

## The idea

### The problem

The readings answer two kinds of question. One is about each station, such as Bergen's mean for the
year. The other is about each hour, such as the mean across all four stations at 03:00 on 8 January.
Both are means of the same 35,040 numbers, grouped in two different ways, and `axis` is the argument
that chooses the grouping. Reshaped into stations by days by hours, the readings allow more
groupings, such as each station's daily means, each station's mean for each hour of the day, or the
network's mean for each day.

A wrong choice between `axis=0` and `axis=1` usually runs without an error, because both choices give
an array of numbers. One of the Common errors below asks for four station means, gets the wrong axis,
and prints four temperatures that look entirely reasonable.

### What the axis argument is

> The **axis** argument of a reduction names the dimension to combine, and that dimension disappears
> from the result. On readings of shape (4, 8760), `axis=1` combines the 8,760 hours and leaves one
> value for each of the 4 stations, shape (4,), and `axis=0` combines the 4 stations and leaves one
> value for each hour, shape (8760,).

### Why it works that way

An axis is named by its position in the shape, counted from 0. Read the shape, cross out the axis you
pass, and what remains is the shape of the answer, so (4, 8760) with axis 1 crossed out is (4,). The
mistake to avoid is thinking of the axis as the dimension that stays. The stations stay when the
hours are combined, and so "one value per station" is `axis=1`, the hours' position.

A question says which dimensions should stay. "Each station's mean" keeps the stations, so the hours
go, with `axis=1`. "The network's mean at each hour" keeps the hours, so the stations go, with
`axis=0`. On (4, 365, 24), "each station's mean for each hour of the day" keeps the stations and the
hours of the day, so the days go, with `axis=1`, and the result is (4, 24). A tuple names several
dimensions at once, and `axis=(1, 2)` combines the days and the hours to leave one value per station.

Negative numbers count from the end, so `axis=-1` is always the last dimension, which is the hours
in (4, 8760) and the hour of the day in (4, 365, 24). `keepdims=True` keeps the combined dimension with
length 1 instead of removing it, which is what broadcasting needs in order to subtract the result
from the array it came from.

### Where this shows up

The **Pandas, Deep Dive** guide's reductions take `axis` too, where `axis=0` combines the rows and
gives one value per column, the same rule applied to a table. An image is an array of shape (height,
width, 3), and a mean over `axis=(0, 1)` gives its average color. Machine learning puts samples on
axis 0, so a mean over `axis=0` gives one value per feature.

### What this notebook covers

- the shape rule, in which the named axis disappears
- `axis=1` for one value per station, and `axis=0` for one value per hour
- `all` and `any` along an axis
- tuples of axes on the readings reshaped to (4, 365, 24), and negative axes
- `argmax` along an axis, for the hour of the day at which each day peaked
- `keepdims=True`, to subtract a result from its array

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np

grid = np.array([[1.0, 2.0, 3.0],
                 [4.0, 5.0, 6.0]])
print(grid.sum(axis=0))
print(grid.sum(axis=1))
print(grid.sum())
```

```
[5. 7. 9.]
[ 6. 15.]
21.0
```

`axis=0` combined the two rows and left one sum for each column. `axis=1` combined the three columns
and left one sum for each row, and with no axis every value went into a single sum.

## Setup

Three imports, the year of readings, and a helper that finds an hour's position.

- `numpy`, imported as `np`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings, and `hour_of_year` finds positions

`readings_list` holds four lists of 8,760 readings, one for each station in `STATIONS`, made by
`year_of_readings`, the formula the **Why NumPy** notebook explained, and `readings` is the same
values as one array of shape (4, 8760), with `nan` for Svalbard's silent day. `hour_of_year(7, 1, 12)`
gives the position of noon on 1 July along a station's row.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


## Worked examples

### The named axis disappears

In [2]:
print(readings.shape)
print(readings.sum(axis=0).shape)
print(readings.sum(axis=1).shape)
print(readings.sum().shape)

(4, 8760)
(8760,)
(4,)
()


Crossing out axis 0 of (4, 8760) leaves (8760,), and crossing out axis 1 leaves (4,). With no axis
every dimension goes, and the shape `()` belongs to a single number.

### axis=1 for one value per station

In [3]:
station_means = np.nanmean(readings, axis=1)

for name, mean in zip(STATIONS, station_means):
    print(f"{name:8} {mean:6.2f}")

Bergen     8.00
Oslo       6.50
Svalbard  -4.48
Tromso     3.50


The hours were combined, and each station kept a mean of its own.

### axis=0 for one value per hour

With `axis=0` the four stations are combined, so every hour of the year gets the network's mean:

In [4]:
network = np.nanmean(readings, axis=0)
coldest = np.nanargmin(network)

print(network.shape)
print(readings[:, 0], network[0].round(2))
print(START + timedelta(hours=int(coldest)), network[coldest].round(2))
print(readings[:, coldest])

(8760,)
[ -3.6  -3.5 -14.6  -6.7] -7.1
2025-01-08 03:00:00 -9.18
[ -4.4  -6.  -17.1  -9.2]


At the first hour of the year the four readings averaged -7.1 degrees. The network as a whole was
coldest at 03:00 on 8 January, at -9.18 degrees, the mean of the four readings in the last line. At
the hours Svalbard missed, `np.nanmean` took the mean of the three stations that reported:

In [5]:
noon_march_2 = hour_of_year(3, 2, 12)

print(readings[:, noon_march_2], network[noon_march_2].round(2))

[ 3.2  1.6  nan -1.6] 1.07


### all and any along an axis

A mask reduces along an axis too. `sum` counts the `True` values, `all` asks whether every value
along the axis is `True`, and `any` asks whether at least one is:

In [6]:
below = readings < 0
stations_below = below.sum(axis=0)

print(stations_below[:6])
print(np.bincount(stations_below))
print(below.all(axis=0).sum(), below.any(axis=0).sum())
print(below.any(axis=1))

[4 4 4 4 4 4]
[2868 2699 1343  643 1207]
1207 5892
[ True  True  True  True]


`stations_below` counts, for every hour, the stations below freezing, from 0 to 4. `np.bincount`
counts how often each of those numbers occurs, so 2,868 hours had no station below freezing and 1,207
had all four. `all(axis=0)` found those same 1,207 hours, and `any(axis=0)` found 5,892 with at least
one station below freezing. Along `axis=1`, `any` asks the question of each station's whole year,
and every station froze at least once.

### Tuples of axes in three dimensions

Reshaped to (4, 365, 24), the readings have a dimension for the station, one for the day and one for
the hour of the day. Every choice of axes crosses out its own dimensions:

In [7]:
by_day = readings.reshape(4, 365, 24)

for axis in [0, 1, 2, (1, 2), (0, 1)]:
    print(f"axis={axis!s:7} {by_day.mean(axis=axis).shape}")

axis=0       (365, 24)
axis=1       (4, 24)
axis=2       (4, 365)
axis=(1, 2)  (4,)
axis=(0, 1)  (24,)


`axis=2` gives each station's daily means and `axis=1` each station's mean at each hour of the day.
`axis=(1, 2)` combines the days and the hours, which leaves one mean per station, the same values as
`axis=1` on the original readings:

In [8]:
print(np.nanmean(by_day, axis=(1, 2)).round(2))
print(station_means.round(2))

[ 8.    6.5  -4.48  3.5 ]
[ 8.    6.5  -4.48  3.5 ]


### Negative axes

In [9]:
print(readings.mean(axis=-1).shape)
print(by_day.mean(axis=-1).shape)

(4,)
(4, 365)


`axis=-1` is the last dimension of whatever array it is given, which is the hours of the year in
`readings` and the hours of the day in `by_day`. Code that should combine the innermost dimension of
an array of any shape can say so with `-1`.

### argmax along an axis

`argmax(axis=2)` gives, for every station and day, the position of the largest value along the hours,
which is the hour of the day at which that day peaked:

In [10]:
peak_hour = by_day.argmax(axis=2)

print(peak_hour.shape)
print(peak_hour[0, :10])
print(np.bincount(peak_hour[0], minlength=24)[12:18])

(4, 365)
[16 15 14 16 16 15 14 13 16 16]
[  0  64  65  65 169   2]


Bergen's first ten days peaked between 13:00 and 16:00, and over the year 169 of its days peaked at
16:00. The positions count within the axis, from 0 to 23, and are hours of the day, where an
`argmax` over the whole row counts from the start of the year.

### keepdims=True for subtracting a result

With `keepdims=True` the daily means have shape (4, 365, 1), which broadcasts against (4, 365, 24),
so every hour has its own day's mean taken away:

In [11]:
day_means = by_day.mean(axis=2, keepdims=True)
within_day = by_day - day_means
july_1 = hour_of_year(7, 1) // 24

print(day_means.shape, within_day.shape)
print(within_day[0, july_1, [3, 15]].round(2))

(4, 365, 1) (4, 365, 24)
[-3.63  2.57]


On 1 July Bergen was 3.63 degrees below its day's mean at 03:00 and 2.57 above it at 15:00.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/11-the-axis-argument-solutions.ipynb).

**1.** Print each station's largest reading with `np.nanmax` and the axis that keeps the stations, and the shape of the result.

In [12]:
# your code here

**2.** Count, for every hour, how many stations were above 10 degrees, and print how many hours had at least three.

In [13]:
# your code here

**3.** Print each station's coldest daily mean, averaging `by_day` over the hours with `axis=2` and then taking `np.nanmin` along the days.

In [14]:
# your code here

**4.** Use a tuple of axes on `by_day` to find the network's mean at each hour of the day, all four stations and all 365 days together, and print the values for 03:00 and 15:00, rounded to two places.

In [15]:
# your code here

**5.** Subtract each day's mean from its hours with `keepdims=True`, and print the largest value in Tromso's result, row 3, rounded to two places.

In [16]:
# your code here

## Common errors

### AxisError: axis 2 is out of bounds for array of dimension 2

In [17]:
readings.mean(axis=2)

AxisError: axis 2 is out of bounds for array of dimension 2

`readings` has two dimensions, numbered 0 and 1, and `axis=2` names a third that only `by_day` has.
Print an array's `shape` before choosing an axis. The full name of the error is
`numpy.exceptions.AxisError`, and it is a kind of `ValueError` and of `IndexError`, so an `except`
for either one catches it:

In [18]:
print(issubclass(np.exceptions.AxisError, ValueError))
print(issubclass(np.exceptions.AxisError, IndexError))

True
True


### ValueError: operands could not be broadcast together with shapes (4,365,24) (4,365)

In [19]:
by_day - by_day.mean(axis=2)

ValueError: operands could not be broadcast together with shapes (4,365,24) (4,365) 

Without `keepdims`, axis 2 disappeared and the daily means came out as (4, 365). Lined up on the
right, as the **Broadcasting** notebook does, that compares 365 with 24, which do not fit. With
`keepdims=True` the means are (4, 365, 1), and the 1 stretches along the hours.

### No error, and four numbers that are not the station means

In [20]:
wrong_means = np.nanmean(readings, axis=0)

print(wrong_means[:4].round(2))
print(wrong_means.shape)

[-7.1  -8.48 -8.48 -8.27]
(8760,)


Asked for the four station means, `axis=0` combined the stations and gave the network's mean at every
hour, and its first four values look like four reasonable temperatures. They are the means at 00:00
to 03:00 on 1 January. The shape gives it away, (8760,) where four station means would be (4,), so
check the shape of a reduction before using its values. The station means come from `axis=1`:

In [21]:
print(np.nanmean(readings, axis=1).round(2))

[ 8.    6.5  -4.48  3.5 ]


### TypeError: Cannot interpret '2' as a data type

In [22]:
np.mean(by_day, 1, 2)

TypeError: Cannot interpret '2' as a data type

`np.mean` takes the axis second and the dtype third, so `np.mean(by_day, 1, 2)` passed 1 as the axis
and 2 as a data type, which it is not. Several axes go in one tuple, `np.mean(by_day, axis=(1, 2))`.

## Recap

- Name in `axis` the dimension to combine, which disappears from the result
- Decide which dimensions the question keeps, and pass the others
- Reduce over several dimensions with a tuple, and over the last one with `axis=-1`
- Use `argmax` along an axis for a position within that axis
- Keep a combined dimension with `keepdims=True` when the result is subtracted from its array

## What is next

The **Random Numbers** notebook makes arrays of random values with a seeded `Generator`, so that a
simulation, a sample or a shuffle gives the same result on every run.

---

&#8592; **Previous:** [Aggregation](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/10-aggregation.ipynb)  &nbsp;·&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Random Numbers](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/12-random-numbers.ipynb) &#8594;
